### Setup

In [ ]:
# %pip installs into the kernel selected in VS Code (safer than !pip)
%pip install -q openai python-dotenv requests

In [ ]:
# Load the OpenAI API key from a .env file (never hard-code keys in a notebook)
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()  # reads .env from the notebook's folder

# Fallback: VS Code shows an input box at the top of the window
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


In [ ]:
# Create the client once and reuse it everywhere
import json
from openai import OpenAI

client = OpenAI()        # picks up OPENAI_API_KEY from the environment
MODEL = "gpt-4.1-mini"   # any tool-calling chat model works, e.g. "gpt-4o-mini", "gpt-5-mini"

### Define Tools

In [ ]:
# Create a weather function/tool - calls the free Open-Meteo API (no API key needed)
import requests

def get_weather(city: str) -> int:
    """Provide weather information with city as input and temperature as output in degree celsius (integer)"""

    # 1. City name -> latitude/longitude
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=10,
    ).json()

    if not geo.get("results"):
        raise ValueError(f"City not found: {city}")

    loc = geo["results"][0]

    # 2. Latitude/longitude -> current temperature
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": loc["latitude"],
            "longitude": loc["longitude"],
            "current": "temperature_2m",
        },
        timeout=10,
    ).json()

    return round(weather["current"]["temperature_2m"])

In [ ]:
# Quick test of the tool on its own (weather data by Open-Meteo.com)
get_weather("Chandigarh")

22

In [ ]:
# Suggest Activity
def suggest_activity(temperature: int) -> str:
    """Suggests activity to do as per Temperature as input"""

    # Logic

    return "Jump on Bed"

In [ ]:
# Create all TOOLS (OpenAI function-calling schema)

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Allows to get weather of a City in degree celsius",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city",
                    }
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "suggest_activity",
            "description": "Suggests activity to do as per Temperature as input",
            "parameters": {
                "type": "object",
                "properties": {
                    "temperature": {
                        "type": "integer",
                        "description": "Temperature in degree celsius",
                    }
                },
                "required": ["temperature"],
            },
        },
    },
]

In [ ]:
# Tool Functions - maps strings to actual functions
TOOL_FUNCTIONS = {
    "get_weather": get_weather,
    "suggest_activity": suggest_activity,
}

### ReAct Loop (Tool Calling Loop)

First, one round done by hand: **LLM call -> tool call -> observation -> LLM call**.

In [ ]:
# Question
QUESTION = "How is the weather in Bangalore?"

In [ ]:
# Create messages
MESSAGES = [
    {"role": "user", "content": QUESTION}
]

In [ ]:
# Call LLM with TOOLS
res = client.chat.completions.create(
    model=MODEL,
    max_completion_tokens=1000,
    tools=TOOLS,
    messages=MESSAGES,
)

In [ ]:
res.choices[0]

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_sGsP0bRTQwLXR7jtcU2r3JGB', function=Function(arguments='{"city":"Bangalore"}', name='get_weather'), type='function')]))

In [ ]:
# Append the assistant message (it contains the tool_calls) to the history
MESSAGES.append(res.choices[0].message)

In [ ]:
MESSAGES

In [ ]:
# Execute tool to send back observation to MODEL

for call in res.choices[0].message.tool_calls:
    tool_id = call.id
    func_name = call.function.name
    _args = json.loads(call.function.arguments)

    tool_output = TOOL_FUNCTIONS[func_name](**_args)  # get_weather(city='Bangalore')

    # OpenAI tool message: role + tool_call_id + content
    MESSAGES.append(
        {"role": "tool", "tool_call_id": tool_id, "content": str(tool_output)}
    )

    print(tool_id, func_name, _args, tool_output)

call_sGsP0bRTQwLXR7jtcU2r3JGB get_weather {'city': 'Bangalore'} 30


In [ ]:
MESSAGES

In [ ]:
# Call LLM again to process the tool output
res = client.chat.completions.create(
    model=MODEL,
    max_completion_tokens=1000,
    tools=TOOLS,
    messages=MESSAGES,
)

In [ ]:
res.choices[0].message.content

'The current temperature in Bangalore is 30 degrees Celsius. Would you like me to suggest some activities based on this temperature?'

### Full Tool Calling Loop

In [ ]:
# Tool Calling LOOP

# Question
QUESTION = "Based on weather suggest me an activity in Bengaluru?"

# Create messages
MESSAGES = [{"role": "user", "content": QUESTION}]

# Max Steps
MAX_STEPS = 5

for step in range(MAX_STEPS):
    print(f"==== Step {step} ====")

    # Call LLM
    res = client.chat.completions.create(
        model=MODEL,
        max_completion_tokens=1000,
        tools=TOOLS,
        messages=MESSAGES,
        parallel_tool_calls=False,  # one tool call per step
    )

    message = res.choices[0].message

    # Append Assistant message
    MESSAGES.append(message)

    # Break if no tool calls
    if not message.tool_calls:
        print(f"Final Output: {message.content}")
        break

    # Tool Call
    for call in message.tool_calls:
        tool_id = call.id
        func_name = call.function.name
        _args = json.loads(call.function.arguments)

        tool_output = TOOL_FUNCTIONS[func_name](**_args)

        MESSAGES.append(
            {"role": "tool", "tool_call_id": tool_id, "content": str(tool_output)}
        )

        print("Tool call -> ", tool_id, func_name, _args, tool_output)
else:
    print("Stopped: reached MAX_STEPS without a final answer")

==== Step 0 ====
Tool call ->  call_c45u2V1MHBzs31UVzyg2teHS get_weather {'city': 'Bengaluru'} 26
==== Step 1 ====
Tool call ->  call_30vwBL900TeqXBtNZmHFGYs1 suggest_activity {'temperature': 26} Jump on Bed
==== Step 2 ====
Final Output: The temperature in Bengaluru is currently 26 degrees Celsius. A good activity to do at this temperature is to jump on the bed. Would you like another suggestion or activity for a different temperature?
